In [6]:
import pandas as pd
import sqlite3
from pandas.plotting import scatter_matrix

In [7]:
connect = sqlite3.connect('data/checking-logs.sqlite')

## Посмотрим, какие таблицы есть в базе test и control это таблицы, которые мы сами создали (выборки)

In [8]:
a = connect.cursor().execute('SELECT name from sqlite_master where type= "table"')
print(a.fetchall())

[('pageviews',), ('checker',), ('deadlines',), ('test',), ('control',)]


In [9]:
query = '''
SELECT uid, COUNT(*) AS num_commits
FROM checker
WHERE uid LIKE 'user_%'
AND labname <> 'project1'
GROUP BY uid
'''
commits = pd.io.sql.read_sql(query, connect)
commits

,uid,num_commits
0,user_0,3
1,user_1,62
2,user_10,20
3,user_11,7
4,user_12,86
5,user_13,52
6,user_14,61
7,user_15,23
8,user_16,41
9,user_17,51


## Создадим таблицу с количеством просмотров

In [10]:
query = '''
SELECT uid, COUNT(*) AS num_views
FROM pageviews
WHERE uid LIKE 'user_%'
GROUP BY uid
'''
num_views = pd.io.sql.read_sql(query, connect)
num_views

,uid,num_views
0,user_1,28
1,user_10,89
2,user_14,143
3,user_17,47
4,user_18,3
5,user_19,16
6,user_21,10
7,user_25,179
8,user_28,149
9,user_3,317


#### Создадим таблицу с дельтой: разница между первым коммитом и дедлайном лаборатории (подробно, как мы это делали в Day06 ех03). (Не принимать во внимание project1 для расчета средней разницы и количества коммитов)

In [11]:
query = '''
SELECT uid,
       CAST((JulianDay(test.first_commit_ts) -
            JulianDay(DATETIME(deadlines.deadlines, 'unixepoch'))
                  ) * 24 AS Integer
           ) AS delta
FROM test
LEFT JOIN deadlines ON test.labname=deadlines.labs
WHERE labname <> 'project1'
'''
delta = pd.io.sql.read_sql(query, connect)
delta

,uid,delta
0,user_30,-202
1,user_30,-201
2,user_14,-200
3,user_14,-193
4,user_19,-148
5,user_25,-148
6,user_21,-126
7,user_21,-99
8,user_28,-98
9,user_17,-81


In [12]:
connect.close()